# 🏠 JKUAT Smart Home Energy Management System
## Phase 3 – mBERT Intent Classifier Fine-Tuning
**Model:** `bert-base-multilingual-cased`  
**Intents:** `TURN_ON` | `TURN_OFF` | `GET_STATUS` | `GET_ADVICE`  
**Output:** `saved_relay_model/` → ready for offline deployment on Raspberry Pi 5

---
### ⚡ Before you start
1. Set runtime to **GPU**: `Runtime → Change runtime type → T4 GPU → Save`
2. Run cells **top to bottom**, one at a time
3. Upload your CSV files when Cell 3 prompts you


## Cell 1 — Install & Verify Dependencies

In [1]:
# Install required libraries (Colab already has PyTorch; this upgrades transformers/datasets)
!pip install -q --upgrade transformers datasets scikit-learn evaluate accelerate

# Verify versions
import torch, transformers, datasets, sklearn
print(f"{'PyTorch':<15}: {torch.__version__}")
print(f"{'Transformers':<15}: {transformers.__version__}")
print(f"{'Datasets':<15}: {datasets.__version__}")
print(f"{'Scikit-Learn':<15}: {sklearn.__version__}")
print(f"{'CUDA available':<15}: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"{'GPU':<15}: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00
PyTorch        : 2.10.0+cu128
Transformers   : 5.9.0
Datasets       : 4.8.5
Scikit-Learn   : 1.8.0
CUDA available : True
GPU            : Tesla T4


## Cell 2 — Mount Google Drive
The trained model will be saved to your Drive so it **persists after the session ends**.  
*(Colab's local `/content/` folder is wiped when the session disconnects.)*


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
# ── Edit this path if you want a different folder in your Drive ──
DRIVE_SAVE_DIR = '/content/drive/MyDrive/JKUAT_SmartHome/saved_relay_model'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"✅  Drive mounted. Model will be saved to:\n    {DRIVE_SAVE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅  Drive mounted. Model will be saved to:
    /content/drive/MyDrive/JKUAT_SmartHome/saved_relay_model


## Cell 3 — Upload Your CSV Files
Upload both `relay_train.csv` and `relay_test.csv` from your local machine.  
They will be placed in `/content/` (Colab's working directory).


In [6]:
from google.colab import files
from pathlib import Path

print("📂 Select relay_train.csv and relay_test.csv (you can select both at once)...")
uploaded = files.upload()

# Confirm both files arrived
for fname in ['relay_train.csv', 'relay_test.csv']:
    fpath = Path(f'/content/{fname}')
    if fpath.exists():
        print(f"  ✅  {fname}  ({fpath.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"  ❌  {fname} NOT found — please re-upload it.")


📂 Select relay_train.csv and relay_test.csv (you can select both at once)...


Saving relay_train.csv to relay_train.csv
Saving relay_test.csv to relay_test.csv
  ✅  relay_train.csv  (28.8 KB)
  ✅  relay_test.csv  (7.2 KB)


## Cell 4 — Configuration
All tuneable settings are here. Edit this cell if you want to adjust epochs, batch size, etc.


In [7]:
import os, logging, time
import numpy as np
import torch
from pathlib import Path

# ─── Colab working paths ────────────────────────────────────────────────────
CONTENT_DIR   = '/content'                          # where CSVs live
LOCAL_OUT_DIR = '/content/saved_relay_model'        # local checkpoint dir
# DRIVE_SAVE_DIR was set in Cell 2

CFG = {
    # ── Data ──────────────────────────────────────────────────────────────
    "train_csv"    : f"{CONTENT_DIR}/relay_train.csv",
    "test_csv"     : f"{CONTENT_DIR}/relay_test.csv",
    "text_col"     : "user_input",
    "label_col"    : "label",

    # ── Model ──────────────────────────────────────────────────────────────
    "base_model"   : "bert-base-multilingual-cased",
    "num_labels"   : 4,
    "id2label"     : {0: "TURN_ON", 1: "TURN_OFF",
                      2: "GET_STATUS", 3: "GET_ADVICE"},
    "label2id"     : {"TURN_ON": 0, "TURN_OFF": 1,
                      "GET_STATUS": 2, "GET_ADVICE": 3},

    # ── Tokenizer ──────────────────────────────────────────────────────────
    "max_length"   : 64,           # relay commands are short phrases

    # ── Training ───────────────────────────────────────────────────────────
    "output_dir"   : LOCAL_OUT_DIR,
    "drive_dir"    : DRIVE_SAVE_DIR,   # final save destination
    "epochs"       : 5,
    "train_batch"  : 32,   # GPU T4 can handle 32 comfortably; use 16 if OOM
    "eval_batch"   : 64,
    "warmup_ratio" : 0.1,
    "weight_decay" : 0.01,
    "learning_rate": 3e-5,
    "seed"         : 42,

    # ── Early stopping ──────────────────────────────────────────────────────
    "early_stopping_patience": 2,

    # ── Target ──────────────────────────────────────────────────────────────
    "target_macro_f1": 0.85,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
print("✅  Configuration loaded.")
print(f"   Train CSV : {CFG['train_csv']}")
print(f"   Test  CSV : {CFG['test_csv']}")
print(f"   Output    : {CFG['output_dir']}")
print(f"   Drive save: {CFG['drive_dir']}")


✅  Configuration loaded.
   Train CSV : /content/relay_train.csv
   Test  CSV : /content/relay_test.csv
   Output    : /content/saved_relay_model
   Drive save: /content/drive/MyDrive/JKUAT_SmartHome/saved_relay_model


## Cell 5 — Logging Setup

In [8]:
log_path = f"{CONTENT_DIR}/training_log.txt"

# Remove any existing handlers so re-running the cell works cleanly
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(log_path, mode="w"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
print(f"✅  Logger ready. Log file: {log_path}")


✅  Logger ready. Log file: /content/training_log.txt


## Cell 6 — Environment & GPU Check

In [9]:
def check_environment():
    logger.info("=" * 60)
    logger.info("ENVIRONMENT CHECK")
    logger.info("=" * 60)
    logger.info(f"  PyTorch   : {torch.__version__}")
    cuda_ok = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "N/A"
    logger.info(f"  CUDA      : {cuda_ok}  |  GPU: {gpu_name}")
    device = "cuda" if cuda_ok else "cpu"
    logger.info(f"  Device    : {device.upper()}")
    if device == "cpu":
        logger.warning("  ⚠️  Running on CPU – training will be SLOW (~20-40 min/epoch).")
        logger.warning("     Go to Runtime → Change runtime type → T4 GPU and re-run.")
    logger.info("=" * 60)
    return device

DEVICE = check_environment()


2026-05-20 20:44:41,117  INFO  ============================================================
2026-05-20 20:44:41,119  INFO  ENVIRONMENT CHECK
2026-05-20 20:44:41,120  INFO  ============================================================
2026-05-20 20:44:41,121  INFO    PyTorch   : 2.10.0+cu128
2026-05-20 20:44:41,123  INFO    CUDA      : True  |  GPU: Tesla T4
2026-05-20 20:44:41,124  INFO    Device    : CUDA
2026-05-20 20:44:41,125  INFO  ============================================================


## Cell 7 — Load & Validate Datasets

In [26]:
from datasets import load_dataset, DatasetDict

def load_and_validate_data() -> DatasetDict:
    logger.info("Loading datasets …")

    for csv_file in [CFG["train_csv"], CFG["test_csv"]]:
        if not Path(csv_file).exists():
            raise FileNotFoundError(
                f"[ERROR] Cannot find '{csv_file}'.\n"
                f"  ➜ Re-run Cell 3 and upload both CSV files."
            )

    raw = load_dataset(
        "csv",
        data_files={"train": CFG["train_csv"], "test": CFG["test_csv"]},
    )

    # Validate columns
    required_cols = {"user_input", "intent", "language", "label"}
    for split, ds in raw.items():
        missing = required_cols - set(ds.column_names)
        if missing:
            raise ValueError(f"[ERROR] Split '{split}' missing columns: {missing}")

    # Validate label values
    for split, ds in raw.items():
        bad = [x for x in ds[CFG["label_col"]] if x not in range(CFG["num_labels"])]
        if bad:
            raise ValueError(f"[ERROR] Split '{split}' has unexpected labels: {set(bad)}")

    # Print class distribution
    for split, ds in raw.items():
        labels = ds[CFG["label_col"]]
        logger.info(f"  {split} set  →  {len(ds)} samples")
        for lid, lname in CFG["id2label"].items():
            logger.info(f"      {lname:<12} (label {lid})  :  {labels.count(lid)} samples")

    return raw

RAW_DATASETS = load_and_validate_data()


2026-05-20 21:32:41,991  INFO  Loading datasets …
2026-05-20 21:32:42,224  INFO  HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/csv/csv.py "HTTP/1.1 200 OK"


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

2026-05-20 21:32:42,348  INFO    train set  →  648 samples
2026-05-20 21:32:42,360  INFO        TURN_ON      (label 0)  :  163 samples
2026-05-20 21:32:42,370  INFO        TURN_OFF     (label 1)  :  165 samples
2026-05-20 21:32:42,379  INFO        GET_STATUS   (label 2)  :  160 samples
2026-05-20 21:32:42,389  INFO        GET_ADVICE   (label 3)  :  160 samples
2026-05-20 21:32:42,389  INFO    test set  →  160 samples
2026-05-20 21:32:42,394  INFO        TURN_ON      (label 0)  :  40 samples
2026-05-20 21:32:42,397  INFO        TURN_OFF     (label 1)  :  40 samples
2026-05-20 21:32:42,401  INFO        GET_STATUS   (label 2)  :  40 samples
2026-05-20 21:32:42,405  INFO        GET_ADVICE   (label 3)  :  40 samples


## Cell 8 — Tokenize Inputs

In [27]:
from transformers import AutoTokenizer

def build_tokenizer_and_tokenize(raw: DatasetDict):
    logger.info(f"\nLoading tokenizer: {CFG['base_model']} …")
    tokenizer = AutoTokenizer.from_pretrained(CFG["base_model"])

    def tokenize_fn(batch):
        return tokenizer(
            batch[CFG["text_col"]],
            padding="max_length",
            truncation=True,
            max_length=CFG["max_length"],
        )

    logger.info("Tokenizing …")
    tokenized = raw.map(tokenize_fn, batched=True, desc="Tokenizing")
    tokenized = tokenized.rename_column(CFG["label_col"], "labels")
    tokenized.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "token_type_ids", "labels"],
    )
    logger.info("  Tokenization complete.")
    return tokenizer, tokenized

TOKENIZER, TOKENIZED = build_tokenizer_and_tokenize(RAW_DATASETS)


2026-05-20 21:32:50,411  INFO  
Loading tokenizer: bert-base-multilingual-cased …
2026-05-20 21:32:50,572  INFO  HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-05-20 21:32:50,660  INFO  HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-20 21:32:50,748  INFO  HTTP Request: GET https://huggingface.co/api/models/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-20 21:32:50,832  INFO  HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-multilingual-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-20 21:32:50,919  INFO  HTTP Request: GET https://huggingface.co/api/models/bert-base-multilingual-cased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-2

Tokenizing:   0%|          | 0/648 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/160 [00:00<?, ? examples/s]

2026-05-20 21:32:53,386  INFO    Tokenization complete.


## Cell 9 — Load mBERT Model

In [28]:
from transformers import AutoModelForSequenceClassification

def build_model():
    logger.info(f"\nLoading model: {CFG['base_model']} …")
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG["base_model"],
        num_labels=CFG["num_labels"],
        id2label=CFG["id2label"],
        label2id=CFG["label2id"],
    )
    total   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"  Total params    : {total:,}")
    logger.info(f"  Trainable params: {trainable:,}")
    return model

MODEL = build_model()


2026-05-20 21:32:55,061  INFO  
Loading model: bert-base-multilingual-cased …
2026-05-20 21:32:55,150  INFO  HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-05-20 21:32:55,241  INFO  HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-05-20 

## Cell 10 — Define Metrics

In [29]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"macro_f1": macro_f1}

print("✅  Metrics function defined (Macro F1 via Scikit-Learn).")


✅  Metrics function defined (Macro F1 via Scikit-Learn).


## Cell 11 — Training Configuration

In [30]:
from transformers import TrainingArguments

def build_training_args(device: str) -> TrainingArguments:
    use_fp16 = (device == "cuda")   # fp16 = faster on GPU, not supported on CPU
    return TrainingArguments(
        output_dir                  = CFG["output_dir"],
        num_train_epochs            = CFG["epochs"],
        per_device_train_batch_size = CFG["train_batch"],
        per_device_eval_batch_size  = CFG["eval_batch"],
        warmup_ratio                = CFG["warmup_ratio"],
        weight_decay                = CFG["weight_decay"],
        learning_rate               = CFG["learning_rate"],

        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "macro_f1",
        greater_is_better           = True,

        logging_dir                 = os.path.join(CFG["output_dir"], "logs"),
        logging_steps               = 10,
        report_to                   = "none",   # disables wandb / comet prompts

        fp16                        = use_fp16,
        dataloader_num_workers      = 2,        # Colab supports multi-worker loading
        seed                        = CFG["seed"],
    )

TRAINING_ARGS = build_training_args(DEVICE)
print(f"✅  Training args ready.")
print(f"   Epochs       : {CFG['epochs']}")
print(f"   Train batch  : {CFG['train_batch']}  |  Eval batch: {CFG['eval_batch']}")
print(f"   Learning rate: {CFG['learning_rate']}")
print(f"   FP16 (GPU)   : {TRAINING_ARGS.fp16}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅  Training args ready.
   Epochs       : 5
   Train batch  : 32  |  Eval batch: 64
   Learning rate: 3e-05
   FP16 (GPU)   : True


## Cell 12 — Fine-Tune the Model ⏳
This is the main training cell. On a **T4 GPU** with a typical dataset size (few hundred samples),  
expect **~2–8 minutes total**. On CPU expect 20–40 min per epoch.


In [31]:
from transformers import Trainer, EarlyStoppingCallback, set_seed

set_seed(CFG["seed"])
start_time = time.time()

trainer = Trainer(
    model           = MODEL,
    args            = TRAINING_ARGS,
    train_dataset   = TOKENIZED["train"],
    eval_dataset    = TOKENIZED["test"],
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(
                           early_stopping_patience=CFG["early_stopping_patience"]
                       )],
)

logger.info("\nStarting fine-tuning …")
trainer.train()
logger.info(f"\nTraining complete in {(time.time()-start_time)/60:.1f} min.")


2026-05-20 21:33:04,397  INFO  
Starting fine-tuning …


Epoch,Training Loss,Validation Loss,Macro F1
1,0.897581,0.548288,0.881501
2,0.245994,0.069133,1.000000
3,0.030433,0.014839,1.000000
4,0.017758,0.010022,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

## Cell 13 — Detailed Evaluation Report

In [32]:
def detailed_evaluation(trainer, tokenized):
    logger.info("\n" + "=" * 60)
    logger.info("DETAILED EVALUATION ON TEST SET")
    logger.info("=" * 60)

    preds_output = trainer.predict(tokenized["test"])
    y_pred = np.argmax(preds_output.predictions, axis=-1)
    y_true = preds_output.label_ids
    label_names = [CFG["id2label"][i] for i in range(CFG["num_labels"])]

    report   = classification_report(y_true, y_pred, target_names=label_names,
                                     digits=4, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm       = confusion_matrix(y_true, y_pred)

    logger.info(f"\n{report}")
    logger.info(f"Confusion Matrix:\n{cm}")
    logger.info(f"\n{'─'*40}")
    logger.info(f"  Macro F1 Score : {macro_f1:.4f}")
    logger.info(f"  Target         : {CFG['target_macro_f1']:.2f}")
    if macro_f1 >= CFG["target_macro_f1"]:
        logger.info("  ✅  TARGET MET  – model is ready for edge deployment.")
    else:
        logger.info("  ⚠️   BELOW TARGET – consider more data or extra epochs.")
    logger.info(f"{'─'*40}")
    return macro_f1

MACRO_F1 = detailed_evaluation(trainer, TOKENIZED)


2026-05-20 21:36:34,799  INFO  
2026-05-20 21:36:34,801  INFO  DETAILED EVALUATION ON TEST SET
2026-05-20 21:36:34,802  INFO  ============================================================


2026-05-20 21:36:35,148  INFO  
              precision    recall  f1-score   support

     TURN_ON     1.0000    1.0000    1.0000        40
    TURN_OFF     1.0000    1.0000    1.0000        40
  GET_STATUS     1.0000    1.0000    1.0000        40
  GET_ADVICE     1.0000    1.0000    1.0000        40

    accuracy                         1.0000       160
   macro avg     1.0000    1.0000    1.0000       160
weighted avg     1.0000    1.0000    1.0000       160

2026-05-20 21:36:35,149  INFO  Confusion Matrix:
[[40  0  0  0]
 [ 0 40  0  0]
 [ 0  0 40  0]
 [ 0  0  0 40]]
2026-05-20 21:36:35,151  INFO  
────────────────────────────────────────
2026-05-20 21:36:35,152  INFO    Macro F1 Score : 1.0000
2026-05-20 21:36:35,153  INFO    Target         : 0.85
2026-05-20 21:36:35,154  INFO    ✅  TARGET MET  – model is ready for edge deployment.
2026-05-20 21:36:35,155  INFO  ────────────────────────────────────────


## Cell 14 — Save Model Locally in Colab (`/content/`)

In [33]:
from pathlib import Path

# Fast tokenizer only needs these 4 files
required = ["config.json", "tokenizer_config.json", "tokenizer.json", "model.safetensors"]

all_ok = True
for fname in required:
    fpath = Path(CFG["output_dir"]) / fname
    ok = fpath.exists()
    print(f"  {'✅' if ok else '❌'} {fname}")
    if not ok:
        all_ok = False

print()
print("✅ Pi-deployment ready!" if all_ok else "❌ Missing files.")

  ✅ config.json
  ✅ tokenizer_config.json
  ✅ tokenizer.json
  ✅ model.safetensors

✅ Pi-deployment ready!


In [34]:
def save_artifacts(trainer, tokenizer, out_dir):
    logger.info(f"\nSaving model & tokenizer to '{out_dir}' …")
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    expected = ["config.json", "tokenizer_config.json",
                "vocab.txt", "special_tokens_map.json"]
    weight_files = (list(Path(out_dir).glob("pytorch_model*.bin")) +
                    list(Path(out_dir).glob("model*.safetensors")))

    all_ok = True
    logger.info("\n  Saved assets:")
    for fname in expected:
        fpath = Path(out_dir) / fname
        status = "✅" if fpath.exists() else "❌  MISSING"
        logger.info(f"    {status}  {fname}")
        if not fpath.exists():
            all_ok = False

    for wf in weight_files:
        size_mb = wf.stat().st_size / 1_048_576
        logger.info(f"    ✅  {wf.name}  ({size_mb:.1f} MB)")
    if not weight_files:
        logger.warning("    ❌  No weight file found!")
        all_ok = False

    if all_ok:
        logger.info(f"\n  ✅  All assets verified in '{out_dir}'.")
    return all_ok

save_artifacts(trainer, TOKENIZER, CFG["output_dir"])


2026-05-20 21:36:42,982  INFO  
Saving model & tokenizer to '/content/saved_relay_model' …


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-20 21:36:46,457  INFO  
  Saved assets:
2026-05-20 21:36:46,459  INFO      ✅  config.json
2026-05-20 21:36:46,459  INFO      ✅  tokenizer_config.json
2026-05-20 21:36:46,460  INFO      ❌  MISSING  vocab.txt
2026-05-20 21:36:46,461  INFO      ❌  MISSING  special_tokens_map.json
2026-05-20 21:36:46,462  INFO      ✅  model.safetensors  (678.5 MB)


False

## Cell 15 — Copy Model to Google Drive 💾
This makes the model **permanent** — it survives after the Colab session ends.  
You can then download it from Drive to your Raspberry Pi.


In [35]:
import shutil

src = CFG["output_dir"]   # /content/saved_relay_model
dst = CFG["drive_dir"]    # /content/drive/MyDrive/JKUAT_SmartHome/saved_relay_model

logger.info(f"Copying model to Google Drive …\n  {src} → {dst}")

# Copy each file individually (more robust than shutil.copytree for Colab Drive)
os.makedirs(dst, exist_ok=True)
copied = []
for f in Path(src).iterdir():
    if f.is_file():
        shutil.copy2(f, dst)
        copied.append(f.name)

logger.info(f"  Copied {len(copied)} files:")
for fname in sorted(copied):
    size_mb = (Path(dst) / fname).stat().st_size / 1_048_576
    logger.info(f"    ✅  {fname}  ({size_mb:.1f} MB)")

logger.info("\n✅  Model is now permanently saved on your Google Drive.")
logger.info(f"   Path: {dst}")


2026-05-20 21:36:54,691  INFO  Copying model to Google Drive …
  /content/saved_relay_model → /content/drive/MyDrive/JKUAT_SmartHome/saved_relay_model
2026-05-20 21:36:57,694  INFO    Copied 5 files:
2026-05-20 21:36:57,697  INFO      ✅  config.json  (0.0 MB)
2026-05-20 21:36:57,699  INFO      ✅  model.safetensors  (678.5 MB)
2026-05-20 21:36:57,700  INFO      ✅  tokenizer.json  (2.8 MB)
2026-05-20 21:36:57,702  INFO      ✅  tokenizer_config.json  (0.0 MB)
2026-05-20 21:36:57,704  INFO      ✅  training_args.bin  (0.0 MB)
2026-05-20 21:36:57,705  INFO  
✅  Model is now permanently saved on your Google Drive.
2026-05-20 21:36:57,706  INFO     Path: /content/drive/MyDrive/JKUAT_SmartHome/saved_relay_model


## Cell 16 — Download as ZIP (Optional)
If you want to download the model directly to your PC instead of (or in addition to) Drive,  
run this cell. A `saved_relay_model.zip` file will download via your browser.


In [37]:
import zipfile
from google.colab import files

zip_path = "/content/saved_relay_model.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in Path(CFG["output_dir"]).iterdir():
        if f.is_file():
            zf.write(f, arcname=f.name)

zip_size = Path(zip_path).stat().st_size / 1_048_576
print(f"✅  ZIP created: {zip_path}  ({zip_size:.1f} MB)")
print("   Downloading to your computer …")
files.download(zip_path)


✅  ZIP created: /content/saved_relay_model.zip  (630.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 17 — Final Asset Verification
Run this at any time to confirm the model directory has all files needed for the Pi.


In [36]:
print("\n" + "="*55)
print("  ASSET VERIFICATION — Pi Deployment Checklist")
print("="*55)

check_dir = CFG["drive_dir"]

# Fast tokenizer doesn't produce vocab.txt or special_tokens_map.json
required = ["config.json", "tokenizer_config.json", "tokenizer.json"]
weight_files = (list(Path(check_dir).glob("pytorch_model*.bin")) +
                list(Path(check_dir).glob("model*.safetensors")))

all_ok = True
for fname in required:
    fpath = Path(check_dir) / fname
    ok = fpath.exists()
    print(f"  {'✅' if ok else '❌'}  {fname}")
    if not ok: all_ok = False

for wf in weight_files:
    size_mb = wf.stat().st_size / 1_048_576
    print(f"  ✅  {wf.name}  ({size_mb:.1f} MB)")
if not weight_files:
    print("  ❌  No weight file found!")
    all_ok = False

print("─"*55)
print(f"  Macro F1 achieved : {MACRO_F1:.4f}")
print(f"  Target            : {CFG['target_macro_f1']:.2f}")
status = "✅  READY for Pi deployment!" if all_ok and MACRO_F1 >= CFG["target_macro_f1"] else "⚠️  Review issues above."
print(f"  Status            : {status}")
print("="*55)


  ASSET VERIFICATION — Pi Deployment Checklist
  ✅  config.json
  ✅  tokenizer_config.json
  ✅  tokenizer.json
  ✅  model.safetensors  (678.5 MB)
───────────────────────────────────────────────────────
  Macro F1 achieved : 1.0000
  Target            : 0.85
  Status            : ✅  READY for Pi deployment!


In [25]:
"""
patch_dataset.py
Appends subjunctive Swahili + reinforcement rows to relay_train.csv
Run this ONCE from your project folder before retraining.
"""

import csv
from pathlib import Path

TRAIN_CSV = "relay_train.csv"

NEW_ROWS = [
    # Subjunctive TURN_OFF (the failing case + variations)
    {"user_input": "Taa izimwe",        "intent": "TURN_OFF", "language": "sw", "label": 1},
    {"user_input": "Taa isiwashwe",     "intent": "TURN_OFF", "language": "sw", "label": 1},
    {"user_input": "Inazimwa taa",      "intent": "TURN_OFF", "language": "sw", "label": 1},
    {"user_input": "Taa ziwe zimwa",    "intent": "TURN_OFF", "language": "sw", "label": 1},
    {"user_input": "Zima taa sasa",     "intent": "TURN_OFF", "language": "sw", "label": 1},
    # Reinforcement TURN_ON
    {"user_input": "Bulb iwashwe",      "intent": "TURN_ON",  "language": "sw", "label": 0},
    {"user_input": "Washa taa sasa",    "intent": "TURN_ON",  "language": "sw", "label": 0},
    {"user_input": "Taa iwashwe sasa",  "intent": "TURN_ON",  "language": "sw", "label": 0},
]

def main():
    csv_path = Path(TRAIN_CSV)

    if not csv_path.exists():
        print(f"❌  '{TRAIN_CSV}' not found.")
        print("   Run this script from the same folder as your CSV files.")
        return

    # Count rows before
    with open(csv_path, "r", encoding="utf-8") as f:
        before = sum(1 for _ in f) - 1  # exclude header

    # Append new rows
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["user_input", "intent", "language", "label"])
        writer.writerows(NEW_ROWS)

    # Count rows after
    with open(csv_path, "r", encoding="utf-8") as f:
        after = sum(1 for _ in f) - 1

    print(f"✅  '{TRAIN_CSV}' updated.")
    print(f"   Rows before : {before}")
    print(f"   Rows added  : {len(NEW_ROWS)}")
    print(f"   Rows after  : {after}")
    print()
    print("Next step: upload the updated relay_train.csv to Colab and retrain from Cell 7.")

if __name__ == "__main__":
    main()

✅  'relay_train.csv' updated.
   Rows before : 640
   Rows added  : 8
   Rows after  : 648

Next step: upload the updated relay_train.csv to Colab and retrain from Cell 7.


In [38]:
!pip install sounddevice scipy

In [39]:
# mic_test.py  — run this, speak for 3 seconds, then check the saved file
import sounddevice as sd
from scipy.io.wavfile import write

SAMPLE_RATE = 16000
DURATION    = 3  # seconds

print("🎤 Recording for 3 seconds — speak now ...")
audio = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE,
               channels=1, dtype='int16')
sd.wait()
write("mic_test.wav", SAMPLE_RATE, audio)
print("✅ Saved to mic_test.wav — play it back to check quality.")

OSError: PortAudio library not found